<a href="https://colab.research.google.com/github/Suman-Poola/Random-Forrest-s-p500/blob/main/Quantum_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from fastai import *
import pandas as pd
from google.colab import userdata
import os
import zipfile
import numpy as np

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

import kaggle


In [2]:
dataset_id = 'darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily'
path = 'ticker_list'
kaggle.api.dataset_download_cli(dataset_id, path = str(path), unzip=True, force=True)


Dataset URL: https://www.kaggle.com/datasets/darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily
License(s): CC0-1.0


100%|██████████| 100M/100M [00:01<00:00, 67.6MB/s] 


In [3]:
!ls {path}

sp500_companies.csv  sp500_stocks.csv


In [4]:
tk = pd.read_csv('ticker_list/sp500_companies.csv')
tk.head()

,symbol,company,sector,sub_industry,headquarters,date_added,founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1989


In [5]:
import numpy as np
tickers = tk['symbol'].tolist()
tickers[0:25]

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AMD',
 'AES',
 'AFL',
 'A',
 'APD',
 'ABNB',
 'AKAM',
 'ALB',
 'ARE',
 'ALGN',
 'ALLE',
 'LNT',
 'ALL',
 'GOOGL',
 'GOOG',
 'MO',
 'AMZN',
 'AMCR',
 'AEE']

In [6]:
len(tickers)

503

In [7]:
tickers[499:503]

['YUM', 'ZBRA', 'ZBH', 'ZTS']

In [8]:
import yfinance as yf
df = yf.download(tickers, period="36mo")
df.head()

/tmp/ipykernel_15134/2136594173.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers, period="36mo")
[*********************100%***********************]  503 of 503 completed


Price            Close                                                 \
Ticker               A        AAPL        ABBV        ABNB        ABT   
Date                                                                    
2023-09-05  116.049271  187.168518  131.721817  142.289993  94.771317   
2023-09-06  115.277092  180.469131  131.280075  141.770004  95.410156   
2023-09-07  113.635017  175.190521  134.255280  143.270004  94.574028   
2023-09-08  111.416275  175.802231  134.354492  145.820007  94.630402   
2023-09-11  110.546364  176.966522  134.372498  147.330002  96.246254   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2023-09-05  72.135162  308.810638  564.880005  173.242661  71.351166  ...   
2023-09-06  72.686684  308.943146  561.940002  173.853394  70.066681  ...   
2023-09-07  74.046471  310.742126  560.460022  170.303467  71.034576  ...   
2023-09-08  73.466423  308.157288  560.359985  169.358673  71.287842  ...   
2023-09-11  74.930809  308.536041  564.500000  169.530457  71.224518  ...   

Price        Volume                                                          \
Ticker           WY     WYNN      XEL       XOM      XYL       XYZ      YUM   
Date                                                                          
2023-09-05  4133700  3084700  4170400  18304000  2679700   5987000  1435900   
2023-09-06  3342700  1896200  3815400  13325500  1514400  10491600  1710200   
2023-09-07  3199500  3111300  3554900  13248500  1807100   9075800  1396800   
2023-09-08  3303400  1466700  3627800  14283200  1690800  20691100   728900   
2023-09-11  2227200  1068600  5586300  14383600  1617400  10420000   981700   

Price                                 
Ticker          ZBH    ZBRA      ZTS  
Date                                  
2023-09-05  1645000  417300  1673000  
2023-09-06  3625200  327200  1356800  
2023-09-07  1816400  498100  1159600  
2023-09-08  1104500  435000  1562200  
2023-09-11  1285900  358400  1069100  

[5 rows x 2515 columns]

In [9]:
df.head()

Price            Close                                                 \
Ticker               A        AAPL        ABBV        ABNB        ABT   
Date                                                                    
2023-09-05  116.049271  187.168518  131.721817  142.289993  94.771317   
2023-09-06  115.277092  180.469131  131.280075  141.770004  95.410156   
2023-09-07  113.635017  175.190521  134.255280  143.270004  94.574028   
2023-09-08  111.416275  175.802231  134.354492  145.820007  94.630402   
2023-09-11  110.546364  176.966522  134.372498  147.330002  96.246254   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2023-09-05  72.135162  308.810638  564.880005  173.242661  71.351166  ...   
2023-09-06  72.686684  308.943146  561.940002  173.853394  70.066681  ...   
2023-09-07  74.046471  310.742126  560.460022  170.303467  71.034576  ...   
2023-09-08  73.466423  308.157288  560.359985  169.358673  71.287842  ...   
2023-09-11  74.930809  308.536041  564.500000  169.530457  71.224518  ...   

Price        Volume                                                          \
Ticker           WY     WYNN      XEL       XOM      XYL       XYZ      YUM   
Date                                                                          
2023-09-05  4133700  3084700  4170400  18304000  2679700   5987000  1435900   
2023-09-06  3342700  1896200  3815400  13325500  1514400  10491600  1710200   
2023-09-07  3199500  3111300  3554900  13248500  1807100   9075800  1396800   
2023-09-08  3303400  1466700  3627800  14283200  1690800  20691100   728900   
2023-09-11  2227200  1068600  5586300  14383600  1617400  10420000   981700   

Price                                 
Ticker          ZBH    ZBRA      ZTS  
Date                                  
2023-09-05  1645000  417300  1673000  
2023-09-06  3625200  327200  1356800  
2023-09-07  1816400  498100  1159600  
2023-09-08  1104500  435000  1562200  
2023-09-11  1285900  358400  1069100  

[5 rows x 2515 columns]

In [10]:
closes = df["Close"]
returns = closes.pct_change() *100

volumes = df['Volume']

cs = closes.stack().rename("Close")
rs = returns.stack().rename("Returns")
vs = volumes.stack().rename("Volume")
fdf = pd.concat([cs,rs,vs], axis = 1).reset_index()

fdf.columns = ['date', 'ticker', 'close', 'return', 'volume']
fdf = fdf.dropna(subset=['return'])

fdf.head(25)



/tmp/ipykernel_15134/4234654860.py:2: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = closes.pct_change() *100


,date,ticker,close,return,volume
495,2023-09-06,A,115.277092,-0.665389,1668900.0
496,2023-09-06,AAPL,180.469131,-3.579334,81755800.0
497,2023-09-06,ABBV,131.280075,-0.335360,5304300.0
498,2023-09-06,ABNB,141.770004,-0.365443,7114400.0
499,2023-09-06,ABT,95.410156,0.674086,5094400.0
500,2023-09-06,ACGL,72.686684,0.764567,922800.0
501,2023-09-06,ACN,308.943146,0.042909,1247900.0
502,2023-09-06,ADBE,561.940002,-0.520465,2299900.0
503,2023-09-06,ADI,173.853394,0.352530,3382500.0
504,2023-09-06,ADM,70.066681,-1.800230,3563900.0


fdf.sort_values(by=['ticker', 'date'], inplace=True)
fdf.head(25)

In [11]:
fdf.sort_values(['ticker', 'date'], inplace=True)
fdf.head(25)

,date,ticker,close,return,volume
495,2023-09-06,A,115.277092,-0.665389,1668900.0
990,2023-09-07,A,113.635017,-1.424459,1317000.0
1485,2023-09-08,A,111.416275,-1.952516,1789800.0
1980,2023-09-11,A,110.546364,-0.780776,2046900.0
2475,2023-09-12,A,110.233604,-0.282921,2035000.0
2970,2023-09-13,A,110.868912,0.576328,2520500.0
3465,2023-09-14,A,112.286179,1.278327,1815800.0
3960,2023-09-15,A,113.292931,0.896595,6889600.0
4455,2023-09-18,A,110.145622,-2.778027,1818000.0
4950,2023-09-19,A,109.774208,-0.337203,2222400.0


In [12]:
df=fdf

In [13]:
len(df)

376041

In [14]:
df['Lag 1'] = df['close'].shift(1)
df['Lag 2'] = df['close'].shift(2)
df['Lag 3'] = df['close'].shift(3)
df['Lag 5'] = df['close'].shift(5)
df.head(200)

,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5
495,2023-09-06,A,115.277092,-0.665389,1668900.0,NaN,NaN,NaN,NaN
990,2023-09-07,A,113.635017,-1.424459,1317000.0,115.277092,NaN,NaN,NaN
1485,2023-09-08,A,111.416275,-1.952516,1789800.0,113.635017,115.277092,NaN,NaN
1980,2023-09-11,A,110.546364,-0.780776,2046900.0,111.416275,113.635017,115.277092,NaN
2475,2023-09-12,A,110.233604,-0.282921,2035000.0,110.546364,111.416275,113.635017,NaN
...,...,...,...,...,...,...,...,...,...
97365,2024-06-14,A,127.596130,-0.551430,2163300.0,128.303635,130.848663,129.816895,130.671783
97864,2024-06-17,A,129.541763,1.524837,2090500.0,127.596130,128.303635,130.848663,130.720932
98363,2024-06-18,A,132.558456,2.328742,3453000.0,129.541763,127.596130,128.303635,129.816895
98862,2024-06-20,A,130.426117,-1.608603,2887400.0,132.558456,129.541763,127.596130,130.848663


In [15]:
df['Lag 1'] = df.groupby('ticker')['close'].shift(1)
df['Lag 2'] = df.groupby('ticker')['close'].shift(2)
df['Lag 3'] = df.groupby('ticker')['close'].shift(3)
df['Lag 5'] = df.groupby('ticker')['close'].shift(5)
df.head(25)

,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5
495,2023-09-06,A,115.277092,-0.665389,1668900.0,NaN,NaN,NaN,NaN
990,2023-09-07,A,113.635017,-1.424459,1317000.0,115.277092,NaN,NaN,NaN
1485,2023-09-08,A,111.416275,-1.952516,1789800.0,113.635017,115.277092,NaN,NaN
1980,2023-09-11,A,110.546364,-0.780776,2046900.0,111.416275,113.635017,115.277092,NaN
2475,2023-09-12,A,110.233604,-0.282921,2035000.0,110.546364,111.416275,113.635017,NaN
2970,2023-09-13,A,110.868912,0.576328,2520500.0,110.233604,110.546364,111.416275,115.277092
3465,2023-09-14,A,112.286179,1.278327,1815800.0,110.868912,110.233604,110.546364,113.635017
3960,2023-09-15,A,113.292931,0.896595,6889600.0,112.286179,110.868912,110.233604,111.416275
4455,2023-09-18,A,110.145622,-2.778027,1818000.0,113.292931,112.286179,110.868912,110.546364
4950,2023-09-19,A,109.774208,-0.337203,2222400.0,110.145622,113.292931,112.286179,110.233604


In [16]:
df['SMA 5'] = df['close'].rolling(window=5).mean()
df['SMA 10'] = df['close'].rolling(window=10).mean()
df['SMA 20'] = df['close'].rolling(window=20).mean()
df['SMA 30'] = df['close'].rolling(window=30).mean()
df['SMA 50'] = df['close'].rolling(window=50).mean()

df.head(100)
#df = fdf[fdf['ticker'] == 'AAPL'].copy()
#df.head()

,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50
495,2023-09-06,A,115.277092,-0.665389,1668900.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
990,2023-09-07,A,113.635017,-1.424459,1317000.0,115.277092,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1485,2023-09-08,A,111.416275,-1.952516,1789800.0,113.635017,115.277092,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1980,2023-09-11,A,110.546364,-0.780776,2046900.0,111.416275,113.635017,115.277092,NaN,NaN,NaN,NaN,NaN,NaN
2475,2023-09-12,A,110.233604,-0.282921,2035000.0,110.546364,111.416275,113.635017,NaN,112.221671,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47595,2024-01-23,A,129.145248,-0.686527,854900.0,130.037994,128.733170,127.987610,128.056274,128.319191,128.174983,130.892303,131.264970,126.024055
48091,2024-01-24,A,127.320473,-1.412964,1141900.0,129.145248,130.037994,128.733170,125.691933,128.644899,128.086683,130.423639,131.356363,126.460070
48587,2024-01-25,A,127.418617,0.077085,1120900.0,127.320473,129.145248,130.037994,127.987610,128.531100,127.967979,129.948129,131.393244,126.883749
49083,2024-01-26,A,128.507568,0.854625,1319800.0,127.418617,127.320473,129.145248,128.733170,128.485980,128.096499,129.526577,131.472299,127.348991


In [17]:
df['volatility 5'] = df['return'].rolling(5).std()
df['volatility 10'] = df['return'].rolling(10).std()
df['volatility 20'] = df['return'].rolling(20).std()
df['volatility 30'] = df['return'].rolling(30).std()
df.head()

,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30
495,2023-09-06,A,115.277092,-0.665389,1668900.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
990,2023-09-07,A,113.635017,-1.424459,1317000.0,115.277092,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1485,2023-09-08,A,111.416275,-1.952516,1789800.0,113.635017,115.277092,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1980,2023-09-11,A,110.546364,-0.780776,2046900.0,111.416275,113.635017,115.277092,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2475,2023-09-12,A,110.233604,-0.282921,2035000.0,110.546364,111.416275,113.635017,NaN,112.221671,NaN,NaN,NaN,NaN,0.663217,NaN,NaN,NaN


In [18]:

df['Price change per volume'] = 1000000*(df['return']/df['volume'])
df.head()


,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30,Price change per volume
495,2023-09-06,A,115.277092,-0.665389,1668900.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.398699
990,2023-09-07,A,113.635017,-1.424459,1317000.0,115.277092,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.081594
1485,2023-09-08,A,111.416275,-1.952516,1789800.0,113.635017,115.277092,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.090913
1980,2023-09-11,A,110.546364,-0.780776,2046900.0,111.416275,113.635017,115.277092,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.381443
2475,2023-09-12,A,110.233604,-0.282921,2035000.0,110.546364,111.416275,113.635017,NaN,112.221671,NaN,NaN,NaN,NaN,0.663217,NaN,NaN,NaN,-0.139028


In [19]:
from sklearn.preprocessing import LabelEncoder

df['id'] = LabelEncoder().fit_transform(df['ticker'])
df.head()


,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30,Price change per volume,id
495,2023-09-06,A,115.277092,-0.665389,1668900.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.398699,0
990,2023-09-07,A,113.635017,-1.424459,1317000.0,115.277092,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.081594,0
1485,2023-09-08,A,111.416275,-1.952516,1789800.0,113.635017,115.277092,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.090913,0
1980,2023-09-11,A,110.546364,-0.780776,2046900.0,111.416275,113.635017,115.277092,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.381443,0
2475,2023-09-12,A,110.233604,-0.282921,2035000.0,110.546364,111.416275,113.635017,NaN,112.221671,NaN,NaN,NaN,NaN,0.663217,NaN,NaN,NaN,-0.139028,0


In [20]:
df['mk cap'] = df['close'] * df['volume'] / 100000000
df.head()

,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,...,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30,Price change per volume,id,mk cap
495,2023-09-06,A,115.277092,-0.665389,1668900.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.398699,0,1.923859
990,2023-09-07,A,113.635017,-1.424459,1317000.0,115.277092,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.081594,0,1.496573
1485,2023-09-08,A,111.416275,-1.952516,1789800.0,113.635017,115.277092,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.090913,0,1.994128
1980,2023-09-11,A,110.546364,-0.780776,2046900.0,111.416275,113.635017,115.277092,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.381443,0,2.262774
2475,2023-09-12,A,110.233604,-0.282921,2035000.0,110.546364,111.416275,113.635017,NaN,112.221671,...,NaN,NaN,NaN,0.663217,NaN,NaN,NaN,-0.139028,0,2.243254


In [21]:
df = df.dropna()
#df = df.drop(columns = 'SMA 100')
df.head()


,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,...,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30,Price change per volume,id,mk cap
24779,2023-11-14,A,109.309952,3.861886,2009700.0,105.245491,106.234673,105.519714,108.262009,106.689107,...,104.620631,106.113876,107.759730,2.200748,2.228398,2.166075,1.988013,1.921623,0,2.196802
25275,2023-11-15,A,111.258949,1.783001,2021400.0,109.309952,105.245491,106.234673,107.135704,107.513756,...,104.864990,106.176884,107.679367,2.160961,2.198582,2.048896,2.000766,0.882062,0,2.248988
25771,2023-11-16,A,111.836800,0.519374,1728100.0,111.258949,109.309952,105.245491,105.519714,108.777173,...,105.094168,106.302247,107.643402,1.781982,2.194612,2.045420,1.987935,0.300546,0,1.932652
26267,2023-11-17,A,110.818214,-0.910778,2700800.0,111.836800,111.258949,109.309952,106.234673,109.693881,...,105.286128,106.384189,107.631441,2.020085,1.887717,2.058794,1.996823,-0.337225,0,2.992978
26763,2023-11-20,A,111.631119,0.733548,3050000.0,110.818214,111.836800,111.258949,105.245491,110.971007,...,105.513347,106.472335,107.653136,1.772125,1.784781,2.062106,1.998321,0.240507,0,3.404749


In [22]:
len(df)

373325

In [23]:
df['future return'] = df.groupby('ticker')['return'].shift(-1)
df = df.dropna()
print(len(df))
df.head(len(df))


/tmp/ipykernel_15134/2419459239.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['future return'] = df.groupby('ticker')['return'].shift(-1)


372822


,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,...,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30,Price change per volume,id,mk cap,future return
24779,2023-11-14,A,109.309952,3.861886,2009700.0,105.245491,106.234673,105.519714,108.262009,106.689107,...,106.113876,107.759730,2.200748,2.228398,2.166075,1.988013,1.921623,0,2.196802,1.783001
25275,2023-11-15,A,111.258949,1.783001,2021400.0,109.309952,105.245491,106.234673,107.135704,107.513756,...,106.176884,107.679367,2.160961,2.198582,2.048896,2.000766,0.882062,0,2.248988,0.519374
25771,2023-11-16,A,111.836800,0.519374,1728100.0,111.258949,109.309952,105.245491,105.519714,108.777173,...,106.302247,107.643402,1.781982,2.194612,2.045420,1.987935,0.300546,0,1.932652,-0.910778
26267,2023-11-17,A,110.818214,-0.910778,2700800.0,111.836800,111.258949,109.309952,106.234673,109.693881,...,106.384189,107.631441,2.020085,1.887717,2.058794,1.996823,-0.337225,0,2.992978,0.733548
26763,2023-11-20,A,111.631119,0.733548,3050000.0,110.818214,111.836800,111.258949,105.245491,110.971007,...,106.472335,107.653136,1.772125,1.784781,2.062106,1.998321,0.240507,0,3.404749,8.720824
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
374027,2026-08-28,ZTS,77.339996,3.092508,4844300.0,75.019997,77.519997,77.290001,77.730003,76.867998,...,75.761666,75.463625,2.267370,2.414346,2.756842,2.371936,0.638381,502,3.746581,-0.310315
374530,2026-08-31,ZTS,77.099998,-0.310315,5798800.0,77.339996,75.019997,77.519997,77.169998,76.853998,...,75.791666,75.442323,2.245722,2.367900,2.757231,2.372857,-0.053514,502,4.470875,0.194555
375033,2026-09-01,ZTS,77.250000,0.194555,4000700.0,77.099998,77.339996,75.019997,77.290001,76.845998,...,75.855000,75.480030,2.246457,2.353867,2.737218,2.362387,0.048630,502,3.090541,-1.398061
375536,2026-09-02,ZTS,76.169998,-1.398061,4474600.0,77.250000,77.099998,77.339996,77.519997,76.575998,...,75.920999,75.477269,2.318983,2.118665,2.708603,2.359121,-0.312444,502,3.408303,0.157546


In [24]:
#features = ['close', 'return', 'Lag 1', 'Lag 2', 'Lag 3', 'Lag 5', 'volatility 5', 'volatility 10', 'volatility 20', 'volatility 30','SMA 10' ,'SMA 20','SMA 30','SMA 50','volume','Price change per volume','id','mk cap']
x = df[features]
y = df['future return']

NameError: name 'features' is not defined

In [ ]:
split_index = int(0.8 * len(df))

x_train, x_test = x.iloc[:split_index], x.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output
!pip install qiskit
!pip install qiskit.circuit
!pip install qiskit.algorithms
!pip install qiskit.quantum_info
!pip install qiskit.primitives
!pip install qiskit_machine_learning
from qiskit import QuantumCircuit as QC
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import z_feature_map
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_machine_learning.optimizers import COBYLA
from qiskit_machine_learning.utils import algorithm_globals
from qiskit_machine_learning.algorithms import NeuralNetworkRegressor
from qiskit_machine_learning.neural_networks import EstimatorQNN
from sklearn.model_selection import train_test_split

algorithm_globals.random_seed = 12345
estimator = Estimator()

In [ ]:
from qiskit import QuantumCircuit as QC


In [ ]:
def conv_circuit(params):
    target = QuantumCircuit(2)
    target.rz(-np.pi / 2, 1)
    target.cx(1, 0)
    target.rz(params[0], 0)
    target.ry(params[1], 1)
    target.cx(0, 1)
    target.ry(params[2], 1)
    target.cx(1, 0)
    target.rz(np.pi / 2, 0)
    return target

def conv_layer(num_qubits, param_prefix):
    qc = QuantumCircuit(num_qubits, name="Convolutional Layer")
    qubits = list(range(num_qubits))
    param_index = 0
    params = ParameterVector(param_prefix, length=num_qubits * 3)
    for q1, q2 in zip(qubits[0::2], qubits[1::2]):
        qc = qc.compose(conv_circuit(params[param_index : (param_index + 3)]), [q1, q2])
        qc.barrier()
        param_index += 3
    for q1, q2 in zip(qubits[1::2], qubits[2::2] + [0]):
        qc = qc.compose(conv_circuit(params[param_index : (param_index + 3)]), [q1, q2])
        qc.barrier()
        param_index += 3

    qc_inst = qc.to_instruction()

    qc = QuantumCircuit(num_qubits)
    qc.append(qc_inst, qubits)
    return qc

In [ ]:
def pool_circuit(params):
    target = QC(2)
    target.rz(-np.pi / 2, 1)
    target.cx(1, 0)
    target.rz(params[0], 0)
    target.ry(params[1], 1)
    target.cx(0, 1)
    target.ry(params[2], 1)

    return target

def pool_layer(sources, sinks, param_prefix):
    num_qubits = len(sources) + len(sinks)
    qc = QC(num_qubits, name="Pooling Layer")
    param_index = 0
    params = ParameterVector(param_prefix, length=num_qubits // 2 * 3)
    for source, sink in zip(sources, sinks):
        qc = qc.compose(pool_circuit(params[param_index : (param_index + 3)]), [source, sink])
        qc.barrier()
        param_index += 3

    qc_inst = qc.to_instruction()

    qc = QC(num_qubits)
    qc.append(qc_inst, range(num_qubits))
    return qc


In [ ]:
features = ['close', 'return', 'Lag 1', 'Lag 2', 'Lag 3', 'Lag 5', 'volatility 5', 'volatility 10', 'volatility 20', 'volatility 30','SMA 10', 'SMA 20', 'SMA 30', 'SMA 50', 'id','volume']
x_train_reduced = x_train.values
x_test_reduced = x_test.values

In [ ]:
len(features)

In [ ]:
x = df[features]
y = df['future return']

split_index = int(0.8 * len(df))
x_train, x_test = x.iloc[:split_index], x.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

In [ ]:

feature_map = z_feature_map(16)

In [ ]:
ansatz = QC(16, name="QCNN")

ansatz.compose(conv_layer(16,'c1'), list(range(16)), inplace=True)
ansatz.compose(pool_layer(list(range(0,8)), list(range(8,16)), "p1"), list(range(16)), inplace=True)

ansatz.compose(conv_layer(8,'c2'), list(range(8,16)), inplace=True)
ansatz.compose(pool_layer(list(range(0,4)), list(range(4,8)), "p2"), list(range(8,16)), inplace=True)

ansatz.compose(conv_layer(4,'c3'), list(range(12,16)), inplace=True)
ansatz.compose(pool_layer(list(range(0,2)), list(range(2,4)), "p3"), list(range(12,16)), inplace=True)

ansatz.compose(conv_layer(2,'c4'), list(range(14,16)), inplace=True)
ansatz.compose(pool_layer(list(range(0,1)), list(range(1,2)), "p4"), list(range(14,16)), inplace=True)

In [ ]:
circuit = QC(16)
circuit.compose(feature_map, range(16), inplace=True)
circuit.compose(ansatz, range(16), inplace=True)

In [ ]:
observable = SparsePauliOp.from_list([("Z" + "I" * 15, 1)])

In [ ]:
qnn = EstimatorQNN(
    circuit=circuit,
    observables=observable,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters,
    estimator=estimator,
)

In [ ]:
regressor = NeuralNetworkRegressor(
    neural_network=qnn,
    optimizer=COBYLA(maxiter=200),
    loss="squared_error",
)

scale_factor = y_train.abs().max()
y_train_scaled = y_train / scale_factor

regressor.fit(x_train_reduced, y_train_scaled)